In [1]:
import os
import subprocess
import numpy as np
import pandas as pd


In [2]:
# This snippet assumes that you run setup first

# This code lists objects in your Google Bucket

# Get the bucket name
my_bucket = os.getenv('WORKSPACE_BUCKET')

# List objects in the bucket
print(subprocess.check_output(f"gsutil ls -r {my_bucket}", shell=True).decode('utf-8'))




gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/bigsnpr_backing/:
gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/bigsnpr_backing/chr10_allind.bk
gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/bigsnpr_backing/chr10_allind.rds
gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/bigsnpr_backing/chr11_allind.bk
gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/bigsnpr_backing/chr11_allind.rds
gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/bigsnpr_backing/chr12_allind.bk
gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/bigsnpr_backing/chr12_allind.rds
gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/bigsnpr_backing/chr13_allind.bk
gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/bigsnpr_backing/chr13_allind.rds
gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/bigsnpr_backing/chr14_allind.bk
gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/bigsnpr_backing/chr14_allind.rds
gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/bigsnpr_backing/chr15_allind.bk
g

In [28]:
%%bash

#!/bin/bash

# Define an array of disease names
diseases=("asthma" "atrial_fibrilation" "breast_cancer" "ckd" "coronary_heart_disease" "hypercholesterolemia"
          "prostate_cancer" "t1d" "t2d")


# Construct the GCS base path
gs_path="gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/dsub/plink"

# Temporary directory to store intermediate files
temp_dir="./temp_combined_files"
mkdir -p "$temp_dir"

# Loop through each disease
for disease in "${diseases[@]}"; do
    echo "Processing Disease: ${disease}"

    # Log file per disease
    log_file="combine_chr_${disease}_log.txt"
    exec > >(tee -a "$log_file") 2>&1

    # Clear file_paths array for each disease
    file_paths=()

    for chromo in {1..22}; do
        file_name="PRS_chr${chromo}_${disease}.profile"
        local_path="${temp_dir}/${file_name}"

        # Try to copy the file from GCS
        if gsutil cp "${gs_path}/${file_name}" "$local_path"; then
            file_paths+=("$local_path")
        else
            echo "Warning: File not found for chr${chromo} - ${file_name}"
        fi
    done

    if [[ ${#file_paths[@]} -eq 0 ]]; then
        echo "No files found for ${disease}, skipping..."
        continue
    fi

    # Combine files using AWK
    awk '
    
        BEGIN { OFS = "," }  # Set output separator to tab
        
        FNR == 1 && NR == 1 { print $2, $4, $5, $6; next }
        FNR == 1 { next }  # Skip headers from all but first file
        {
            cnt[$2] += $4;
            cnt2[$2] += $5;
            score[$2] += $6;
        }
        END {
            for (i in score) {
                print i, cnt[i], cnt2[i], score[i];
            }
        }
    ' "${file_paths[@]}" > "${temp_dir}/combined_${disease}.txt"

    # Upload the combined file to GCS
    gsutil cp "${temp_dir}/combined_${disease}.txt" "gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/data/"

    echo "Done processing ${disease}"
done




Processing Disease: asthma
Copying gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/dsub/plink/PRS_chr1_asthma.profile...
/ [1 files][ 19.0 MiB/ 19.0 MiB]                                                
Operation completed over 1 objects/19.0 MiB.                                     
Copying gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/dsub/plink/PRS_chr2_asthma.profile...
/ [1 files][ 19.0 MiB/ 19.0 MiB]                                                
Operation completed over 1 objects/19.0 MiB.                                     
Copying gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/dsub/plink/PRS_chr3_asthma.profile...
/ [1 files][ 19.0 MiB/ 19.0 MiB]                                                
Operation completed over 1 objects/19.0 MiB.                                     
Copying gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/dsub/plink/PRS_chr4_asthma.profile...
/ [1 files][ 19.0 MiB/ 19.0 MiB]                                                
Operation c

Operation completed over 1 objects/18.2 MiB.                                     
Copying gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/dsub/plink/PRS_chr9_atrial_fibrilation.profile...
/ [1 files][ 18.0 MiB/ 18.0 MiB]                                                
Operation completed over 1 objects/18.0 MiB.                                     
Copying gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/dsub/plink/PRS_chr10_atrial_fibrilation.profile...
/ [1 files][ 18.1 MiB/ 18.1 MiB]                                                
Operation completed over 1 objects/18.1 MiB.                                     
Copying gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/dsub/plink/PRS_chr11_atrial_fibrilation.profile...
/ [1 files][ 18.2 MiB/ 18.2 MiB]                                                
Operation completed over 1 objects/18.2 MiB.                                     
Copying gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/dsub/plink/PRS_chr12_atrial_fibrilati

Copying gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/dsub/plink/PRS_chr16_breast_cancer.profile...
- [1 files][ 18.6 MiB/ 18.6 MiB]                                                
Operation completed over 1 objects/18.6 MiB.                                     
Copying gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/dsub/plink/PRS_chr17_breast_cancer.profile...
/ [1 files][ 18.5 MiB/ 18.5 MiB]                                                
Operation completed over 1 objects/18.5 MiB.                                     
Copying gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/dsub/plink/PRS_chr18_breast_cancer.profile...
/ [1 files][ 18.4 MiB/ 18.4 MiB]                                                
Operation completed over 1 objects/18.4 MiB.                                     
Copying gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/dsub/plink/PRS_chr19_breast_cancer.profile...
/ [1 files][ 18.7 MiB/ 18.7 MiB]                                                
Operat

/ [1 files][ 19.0 MiB/ 19.0 MiB]                                                
Operation completed over 1 objects/19.0 MiB.                                     
Copying gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/dsub/plink/PRS_chr2_coronary_heart_disease.profile...
/ [1 files][ 19.0 MiB/ 19.0 MiB]                                                
Operation completed over 1 objects/19.0 MiB.                                     
Copying gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/dsub/plink/PRS_chr3_coronary_heart_disease.profile...
/ [1 files][ 19.3 MiB/ 19.3 MiB]                                                
Operation completed over 1 objects/19.3 MiB.                                     
Copying gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/dsub/plink/PRS_chr4_coronary_heart_disease.profile...
/ [1 files][ 19.3 MiB/ 19.3 MiB]                                                
Operation completed over 1 objects/19.3 MiB.                                     
Copyin

Copying gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/dsub/plink/PRS_chr8_hypercholesterolemia.profile...
/ [1 files][ 19.0 MiB/ 19.0 MiB]                                                
Operation completed over 1 objects/19.0 MiB.                                     
Copying gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/dsub/plink/PRS_chr9_hypercholesterolemia.profile...
/ [1 files][ 19.1 MiB/ 19.1 MiB]                                                
Operation completed over 1 objects/19.1 MiB.                                     
Copying gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/dsub/plink/PRS_chr10_hypercholesterolemia.profile...
/ [1 files][ 19.0 MiB/ 19.0 MiB]                                                
Operation completed over 1 objects/19.0 MiB.                                     
Copying gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/dsub/plink/PRS_chr11_hypercholesterolemia.profile...
/ [1 files][ 19.3 MiB/ 19.3 MiB]                             

Copying gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/dsub/plink/PRS_chr15_prostate_cancer.profile...
/ [1 files][ 18.2 MiB/ 18.2 MiB]                                                
Operation completed over 1 objects/18.2 MiB.                                     
Copying gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/dsub/plink/PRS_chr16_prostate_cancer.profile...
/ [1 files][ 18.2 MiB/ 18.2 MiB]                                                
Operation completed over 1 objects/18.2 MiB.                                     
Copying gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/dsub/plink/PRS_chr17_prostate_cancer.profile...
/ [1 files][ 18.2 MiB/ 18.2 MiB]                                                
Operation completed over 1 objects/18.2 MiB.                                     
Copying gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/dsub/plink/PRS_chr18_prostate_cancer.profile...
/ [1 files][ 18.2 MiB/ 18.2 MiB]                                               

/ [1 files][ 19.3 MiB/ 19.3 MiB]                                                
Operation completed over 1 objects/19.3 MiB.                                     
Copying gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/dsub/plink/PRS_chr3_t2d.profile...
/ [1 files][ 19.0 MiB/ 19.0 MiB]                                                
Operation completed over 1 objects/19.0 MiB.                                     
Copying gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/dsub/plink/PRS_chr4_t2d.profile...
/ [1 files][ 19.0 MiB/ 19.0 MiB]                                                
Operation completed over 1 objects/19.0 MiB.                                     
Copying gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/dsub/plink/PRS_chr5_t2d.profile...
/ [1 files][ 19.0 MiB/ 19.0 MiB]                                                
Operation completed over 1 objects/19.0 MiB.                                     
Copying gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/dsu

In [27]:
name_of_file_in_bucket = 'combined_asthma.txt'

# copy csv file from the bucket to the current working space
os.system(f"gsutil cp '{my_bucket}/data/{name_of_file_in_bucket}' .")

my_dataframe = pd.read_csv(name_of_file_in_bucket) #, usecols=[0,3])
my_dataframe.head()


Copying gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/data/combined_asthma.txt...
/ [1 files][ 13.0 MiB/ 13.0 MiB]                                                
Operation completed over 1 objects/13.0 MiB.                                     


,IID,CNT,CNT2,SCORE
0,8730010,325724,93265,0.000003
1,4141984,325722,79823,0.000001
2,2723508,325802,78635,0.000002
3,2473401,325724,87996,0.000002
4,1039754,325752,80550,0.000001


In [11]:
name_of_file_in_bucket = 'PRS_chr14_asthma.profile'

# copy csv file from the bucket to the current working space
os.system(f"gsutil cp '{my_bucket}/dsub/plink/{name_of_file_in_bucket}' .")

my_dataframe = pd.read_csv(name_of_file_in_bucket)
my_dataframe.head()

Copying gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/dsub/plink/PRS_chr14_asthma.profile...
/ [1 files][ 19.0 MiB/ 19.0 MiB]                                                
Operation completed over 1 objects/19.0 MiB.                                     


,FID IID PHENO CNT CNT2 SCORE
0,0 1000000 -9 66216 18818 2.11456e-06
1,0 1000004 -9 66222 16269 1.36893e-06
2,0 1000033 -9 66200 15989 9.01139e-07
3,0 1000039 -9 66180 18831 2.09624e-06
4,0 1000042 -9 66212 19191 1.97307e-06


In [18]:
# This snippet assumes that you run setup first

# This code lists objects in your Google Bucket

# Get the bucket name
my_bucket = os.getenv('WORKSPACE_BUCKET')

# List objects in the bucket
print(subprocess.check_output(f"gsutil ls -r {my_bucket}", shell=True).decode('utf-8'))




gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/bigsnpr_backing/:
gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/bigsnpr_backing/chr10_allind.bk
gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/bigsnpr_backing/chr10_allind.rds
gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/bigsnpr_backing/chr11_allind.bk
gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/bigsnpr_backing/chr11_allind.rds
gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/bigsnpr_backing/chr12_allind.bk
gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/bigsnpr_backing/chr12_allind.rds
gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/bigsnpr_backing/chr13_allind.bk
gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/bigsnpr_backing/chr13_allind.rds
gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/bigsnpr_backing/chr14_allind.bk
gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/bigsnpr_backing/chr14_allind.rds
gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/bigsnpr_backing/chr15_allind.bk
g